<a href="https://colab.research.google.com/github/pop756/Quantum_KAN/blob/EMT/Real_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
""" 
!git clone -b EMT https://github.com/pop756/Quantum_KAN.git
%cd Quantum_KAN
!pip install -r requirements.txt """

' \n!git clone -b EMT https://github.com/pop756/Quantum_KAN.git\n%cd Quantum_KAN\n!pip install -r requirements.txt '

In [1]:
def calculate_angle(norm,imag):
    if norm > 0:
    
        sign = 1
    else:
        sign = -1
    real = sign*np.sqrt(norm**2-imag**2)
    angle = np.arctan(imag/real)
    return angle

In [2]:
lee_token = 'caba0c2060852277fdc179f02b2c5829b708c565582cc19953a963b4bde8171e194a356f7723368c8edf97207d9d9a13597a69f4ab3191733bc185822084b678'
sung_token = "e9afc06b7a6fbf0acbad55e28e45203c83e8f9680361a43906ae100816645623f7629f434b8c2e73e7663c280ce061a5b1084d3fca959b7a809da5001ac7b0e5"

In [3]:
# Initialize your account
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit.circuit.library import RealAmplitudes, EfficientSU2

service = QiskitRuntimeService(
    channel="ibm_quantum",
    instance="ibm-q-skku/skku/skku-students",
    token=lee_token,
)
backend = service.backend('ibm_strasbourg')



In [4]:
backend.target['ecr'][(78,77)].calibration.instructions[0][1].pulse._params

{'sigma': 32,
 'width': 472,
 'amp': 0.03423285847646341,
 'angle': -0.002394413238397934}

In [5]:
from functions.Train_ZNE import update_pulse
import json
import numpy as np

with open('./offset_strasbourg_ecr_data_second_x.json','r') as file:
    config_list = json.load(file)
for config in config_list:
    config['amp1'] = config['x_amp_3']
    config['amp2'] = config['second']
    print(config['init'])
    target_pulse_params = backend.target['ecr'][tuple(config['init'])].calibration.instructions[1][1].pulse._params
    angle1 = calculate_angle(config['amp1'],config['y'])
    angle2 = calculate_angle(config['amp2'],config['y'])
    print(target_pulse_params['angle'],config['cr_angle'])
    config['x_angle1'] = angle1
    config['x_angle2'] = angle2
update = update_pulse(backend,config_list)
backend_update = update.update_ecr_real()


[78, 77]
1.3653467710190117 1.3653467710190117
[77, 76]
1.1941250403093424 1.1941250403093424
[79, 78]
2.5739415256288325 2.5739415256288325
[76, 75]
2.2985710921592397 2.2985710921592397


In [6]:
import numpy as np
import numpy as np
from qiskit_experiments.library import StandardRB, InterleavedRB
from qiskit_experiments.framework import ParallelExperiment, BatchExperiment
import qiskit.circuit.library as circuits
import numpy as np
import pandas as pd

## No second x

with open('second_minus_offset_strasbourg.json','r') as file:
    config_list = json.load(file)
for config in config_list:
    config['amp1'] = config['x_amp_3']
    config['amp2'] = config['x_amp_3']
    print(config['init'])
    target_pulse_params = backend.target['ecr'][tuple(config['init'])].calibration.instructions[1][1].pulse._params
    angle1 = calculate_angle(config['amp1'],config['y'])
    angle2 = calculate_angle(config['amp2'],config['y'])
    print(target_pulse_params['angle'],config['cr_angle'])
    config['x_angle1'] = angle1
    config['x_angle2'] = angle2
    print(f"offset : {config['offset']}")
update = update_pulse(backend,config_list)
backend_update = update.update_ecr_real()

lengths = np.arange(1, 100, 10)   
num_samples = 30
seed = 1010

for config in config_list:
    qubits = tuple(config['init'])
    # The interleaved gate is the CX gate
    int_exp2 = InterleavedRB(
        circuits.ECRGate(), qubits, lengths, num_samples=num_samples, seed=seed)
    int_exp2.set_experiment_options(max_circuits=25)
    int_expdata2 = int_exp2.run(backend_update).block_for_results()
    int_results2 = int_expdata2.analysis_results()
    # View result data
    display(int_expdata2.figure(0))
    for result in int_results2:
        print(result)

[78, 77]
1.3653467710190117 1.4121534988026259
offset : 4953.54396888928
[77, 76]
1.1941250403093424 1.1933440592039026
offset : -3323.791119074449
[79, 78]
2.5739415256288325 2.484747877566339
offset : -5653.029758989229
[76, 75]
2.2985710921592397 2.1853918493485036
offset : 21726.91026974935


base_runtime_job._start_websocket_client:WARNING:2024-09-04 22:16:47,327: An error occurred while streaming results from the server for job cvc2ky7p7drg008sfbh0:
Traceback (most recent call last):
  File "C:\Users\pop75\AppData\Roaming\Python\Python311\site-packages\qiskit_ibm_runtime\base_runtime_job.py", line 314, in _start_websocket_client
    self._ws_client.job_results()
  File "C:\Users\pop75\AppData\Roaming\Python\Python311\site-packages\qiskit_ibm_runtime\api\clients\runtime_ws.py", line 70, in job_results
    self.stream(url=url, retries=max_retries, backoff_factor=backoff_factor)
  File "C:\Users\pop75\AppData\Roaming\Python\Python311\site-packages\qiskit_ibm_runtime\api\clients\base_websocket_client.py", line 222, in stream
    raise WebsocketError(error_message)
qiskit_ibm_runtime.api.exceptions.WebsocketError: 'Max retries exceeded: Failed to establish a websocket connection. Error: Traceback (most recent call last):\n  File "c:\\Users\\pop75\\anaconda3\\envs\\EMT\\Lib\\si

In [ ]:
import numpy as np
import numpy as np
from qiskit_experiments.library import StandardRB, InterleavedRB
from qiskit_experiments.framework import ParallelExperiment, BatchExperiment
import qiskit.circuit.library as circuits
import numpy as np
import pandas as pd

## No second x

with open('second_minus_offset_strasbourg.json','r') as file:
    config_list = json.load(file)
for config in config_list:
    config['amp1'] = config['x_amp_3']
    config['amp2'] = config['x_amp_3']
    print(config['init'])
    target_pulse_params = backend.target['ecr'][tuple(config['init'])].calibration.instructions[1][1].pulse._params
    angle1 = calculate_angle(config['amp1'],config['y'])
    angle2 = calculate_angle(config['amp2'],config['y'])
    print(target_pulse_params['angle'],config['cr_angle'])
    config['x_angle1'] = angle1
    config['x_angle2'] = angle2
    print(f"offset : {config['offset']}")
update = update_pulse(backend,config_list)
backend_update = update.update_ecr_real()

lengths = np.arange(1, 100, 10)   
num_samples = 30
seed = 1010

for config in config_list:
    qubits = tuple(config['init'])
    # The interleaved gate is the CX gate
    int_exp2 = InterleavedRB(
        circuits.ECRGate(), qubits, lengths, num_samples=num_samples, seed=seed)
    int_exp2.set_experiment_options(max_circuits=25)
    int_expdata2 = int_exp2.run(backend).block_for_results()
    int_results2 = int_expdata2.analysis_results()
    # View result data
    display(int_expdata2.figure(0))
    for result in int_results2:
        print(result)